In [ ]:
# NOTA: Este notebook reutiliza todo el código de tarea1_3_entrenamiento_es.ipynb
# Solo cambian los archivos de datos y el tokenizador

# Ejecutar primero todas las celdas de arquitectura de tarea1_3 (FeedForward, SelfAttention, etc.)
# O importarlas de un módulo común

%run tarea1_3_entrenamiento_es.ipynb

# Alternativamente, copiar aquí todas las definiciones de clases

## Cargar Tokenizador Inglés

In [ ]:
import pickle

# Cargar tokenizador para inglés
with open('fechas2_tokenizer_en.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

print(f"Tokenizador inglés cargado. Tamaño del vocabulario: {tokenizer.vocab_size}")

## Crear Datasets en Inglés

In [ ]:
# Crear datasets con archivos en inglés
trainset_en = Fechas2ASRDataset(
    '../fechas2/fechas2_train.en.csv',
    tokenizer,
    transform=[NoiseAug(prob=0.5), RIRAug(prob=0.5)]
)

testset_en = Fechas2TestDataset(
    '../fechas2/fechas2_test.en.csv',
    tokenizer
)

## Entrenar Modelo para Inglés

In [ ]:
# Configuración del modelo (igual que español)
model_config = {
    'vocab_size': tokenizer.vocab_size,
    'd_model': 256,
    'nb_layers': 6,
    'd_ff': 512,
    'n_heads': 8,
    'd_head': 32,
    'dropout': 0.1,
    'seq_len': 500,
    'feat_dim': 80
}

model_en = AudioTransformer(**model_config)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_en.to(device)

opt = torch.optim.Adam(model_en.parameters(), lr=3e-4)

nb_epochs = 10
batch_size = 16

trainloader_en = torch.utils.data.DataLoader(
    trainset_en, 
    batch_size=batch_size, 
    shuffle=True
)

print(f"Iniciando entrenamiento para inglés...")

# Entrenamiento
model_en.train()
losses_en = []

for e in range(nb_epochs):
    epoch_loss = 0
    for batch_idx, (x, y) in enumerate(trainloader_en):
        x = x.to(device)
        y = y.to(device)
        
        opt.zero_grad()
        loss = model_en.loss(x, y)
        loss.backward()
        opt.step()
        
        epoch_loss += loss.item()
        
        if (batch_idx + 1) % 100 == 0:
            print(f'  Batch {batch_idx+1}/{len(trainloader_en)}: loss={loss.item():.4f}')
    
    avg_loss = epoch_loss / len(trainloader_en)
    losses_en.append(avg_loss)
    print(f'Epoch {e+1}/{nb_epochs}: avg_loss={avg_loss:.4f}')

torch.save({'model': model_en.state_dict(), 'opt': opt.state_dict(), 'config': model_config}, 
           'model_fechas2_en.pt')
print("Modelo inglés guardado en 'model_fechas2_en.pt'")

## Evaluar Modelo Inglés (WER)

In [ ]:
import jiwer

model_en.eval()
hyp_en = []
ref_en = []

print("Evaluando modelo inglés en test set...")
for i, (x, y, text_orig) in enumerate(testset_en):    
    x = x.to(device)    
    y_pred = model_en.generate(x[None,...])
    
    hyp = tokenizer.decode(y_pred)
    ref = text_orig
    
    hyp_en.append(hyp)
    ref_en.append(ref)
    
    if i < 10:
        print(f"\nEjemplo {i+1}:")
        print(f"  REF: {ref}")
        print(f"  HYP: {hyp}")

# Calcular WER
out_en = jiwer.process_words(ref_en, hyp_en)
print(f"\n{'='*50}")
print(f"WER (English): {out_en.wer:.2%}")
print(f"Sustituciones: {out_en.substitutions}")
print(f"Deleciones: {out_en.deletions}")
print(f"Inserciones: {out_en.insertions}")
print(f"{'='*50}")

# Guardar resultados
results_df_en = pd.DataFrame({
    'reference': ref_en,
    'hypothesis': hyp_en
})
results_df_en.to_csv('results_fechas2_en.csv', index=False)
print("\nResultados guardados en 'results_fechas2_en.csv'")

## Comparar Resultados Español vs Inglés

In [ ]:
# Si ya tienes los resultados en español, puedes compararlos
# Cargar resultados de español si existen
try:
    with open('metrics_fechas2_es.pkl', 'rb') as f:
        metrics_es = pickle.load(f)
    
    print("Comparación Español vs Inglés:")
    print(f"\nWER Español:  {metrics_es['WER']:.2%}")
    print(f"WER Inglés:   {out_en.wer:.2%}")
    print(f"\nMejora/Diferencia: {(metrics_es['WER'] - out_en.wer)*100:.2f} puntos porcentuales")
except:
    print("No se encontraron métricas de español para comparar")